# Session 6: ドローンの数学モデル
# Session 6: Drone Mathematical Model

## 目標 / Objective

6DoF 運動方程式、DC モータモデル、ホバー条件を理解する。

Understand 6DoF equations of motion, DC motor model, and hover condition.

## この Notebook で学ぶこと / What You'll Learn

| トピック | 説明 |
|---------|------|
| 6DoF 運動方程式 | 並進 + 回転のダイナミクス |
| 座標系 | NED座標系 vs 機体座標系 |
| モータモデル | 推力 vs 電圧の関係 |
| ホバー条件 | 推力 = 重力のバランス |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from IPython.display import Math, display

from stampfly_edu.dynamics import (
    derive_equations_of_motion,
    hover_condition,
    linearize_at_hover,
    motor_curve,
)

print("Ready! / 準備完了！")

## 1. 座標系 / Coordinate Systems

### NED 座標系（慣性座標系）
- **N**: 北 (North)
- **E**: 東 (East)  
- **D**: 下 (Down)

### 機体座標系
- **x**: 前方
- **y**: 右方
- **z**: 下方

```
               Front
          FL (M4)   FR (M1)
             ╲   ▲   ╱
              ╲  │  ╱
               ╲ │ ╱
                ╲│╱
                 ╳         ← Center
                ╱│╲
               ╱ │ ╲
              ╱  │  ╲
             ╱   │   ╲
          RL (M3)    RR (M2)
                Rear
```

## 2. 運動方程式の導出 / Deriving Equations of Motion

In [ ]:
# Derive equations symbolically
# 記号的に方程式を導出
eqs = derive_equations_of_motion()

print("=== Rotational Dynamics / 回転ダイナミクス ===")
print("\nd(p)/dt =")
display(Math(sp.latex(eqs['rotational']['dp'])))

print("\nd(q)/dt =")
display(Math(sp.latex(eqs['rotational']['dq'])))

print("\nd(r)/dt =")
display(Math(sp.latex(eqs['rotational']['dr'])))

In [ ]:
# Motor forces and torques
# モーター力とトルク
print("=== Forces and Torques / 力とトルク ===")
for name, expr in eqs['forces'].items():
    print(f"\n{name} =")
    display(Math(sp.latex(expr)))

## 3. ホバー条件 / Hover Condition

In [ ]:
# Calculate hover condition for StampFly
# StampFly のホバー条件を計算
hover = hover_condition()

print("=== Hover Condition / ホバー条件 ===")
print(f"Mass / 質量:            {hover['mass_kg']*1000:.1f} g")
print(f"Total thrust / 合計推力: {hover['total_thrust_N']*1000:.2f} mN")
print(f"Per motor / 1モーター:   {hover['thrust_per_motor_N']*1000:.2f} mN")
print(f"Duty estimate / デューティ: {hover['duty_estimate']*100:.1f} %")
print(f"T/W ratio / 推重比:     {hover['thrust_to_weight_ratio']:.2f}")

## 4. モーターカーブ / Motor Curve

In [ ]:
# Plot motor characteristics
# モーター特性をプロット
mc = motor_curve()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].plot(mc['voltage'], mc['rpm'])
axes[0,0].set_xlabel('Voltage (V) / 電圧')
axes[0,0].set_ylabel('RPM')
axes[0,0].set_title('RPM vs Voltage / RPM vs 電圧')
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(mc['voltage'], mc['thrust_N'] * 1000)
axes[0,1].set_xlabel('Voltage (V) / 電圧')
axes[0,1].set_ylabel('Thrust (mN) / 推力')
axes[0,1].set_title('Thrust vs Voltage / 推力 vs 電圧')
axes[0,1].axhline(y=hover['thrust_per_motor_N']*1000, color='r',
                   linestyle='--', label='Hover / ホバー')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(mc['voltage'], mc['current_A'] * 1000)
axes[1,0].set_xlabel('Voltage (V) / 電圧')
axes[1,0].set_ylabel('Current (mA) / 電流')
axes[1,0].set_title('Current vs Voltage / 電流 vs 電圧')
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(mc['thrust_N'] * 1000, mc['current_A'] * 1000)
axes[1,1].set_xlabel('Thrust (mN) / 推力')
axes[1,1].set_ylabel('Current (mA) / 電流')
axes[1,1].set_title('Current vs Thrust / 電流 vs 推力')
axes[1,1].grid(True, alpha=0.3)

fig.suptitle('StampFly Motor Characteristics / モーター特性', fontsize=14)
fig.tight_layout()
plt.show()

## 5. 線形化 / Linearization

ホバー周りで線形化した状態空間モデルを導出します。

In [ ]:
# Linearize at hover
# ホバーで線形化
lin = linearize_at_hover()

print("=== Roll Axis State Space / ロール軸状態空間 ===")
print(f"State: [phi, p] = [angle, angular rate]")
print(f"\nA =")
print(lin['A_roll'])
print(f"\nB =")
print(lin['B_roll'])

print(f"\nB[1,0] = {lin['B_roll'][1,0]:.1f} rad/s² per unit command")
print(f"This means 1 unit of control input produces {lin['B_roll'][1,0]:.1f} rad/s² angular acceleration")

## 6. 考察課題 / Discussion Questions

1. **推重比の意味**: T/W 比が 1 に近い場合と大きい場合、制御にどう影響するか？
2. **モーターの非線形性**: 推力が電圧に対して非線形なのはなぜか？制御設計にどう影響するか？
3. **ジャイロスコピック効果**: 4つのプロペラの回転方向が交互になっているのはなぜか？

---

1. **T/W Ratio**: How does the control change when T/W is close to 1 vs much larger?
2. **Motor Nonlinearity**: Why is thrust nonlinear with voltage? How does it affect control?
3. **Gyroscopic Effect**: Why do the 4 propellers alternate rotation direction?

In [ ]:
# Your experiments here / ここで実験してみてください
